In [ ]:
!pip install moviepy ffmpeg-python


In [ ]:
import os
import shutil
from moviepy.editor import VideoFileClip
import ffmpeg

def compress_video(input_path, output_path, target_size=30 * 1024 * 1024):
    # Get the original video clip
    clip = VideoFileClip(input_path)
    original_size = os.path.getsize(input_path)

    if original_size <= target_size:
        print(f"{input_path} is already under {target_size / (1024 * 1024)} MB. Skipping compression.")
        
        # Copy the skipped video to the output directory without compression
        os.makedirs(os.path.dirname(output_path), exist_ok=True)
        shutil.copy2(input_path, output_path)  # Copy instead of rename
        
        return "skipped", input_path, original_size  # Mark as skipped
    
    # Calculate the target bitrate in bits per second
    duration = clip.duration
    target_bitrate = (target_size * 8) / duration  # target bitrate in bits per second

    # Set audio bitrate and final video bitrate
    audio_bitrate = 128 * 1024  # You can adjust the audio bitrate (e.g., 128 kbps)
    final_bitrate = max(target_bitrate - audio_bitrate, 100 * 1024)  # Ensure final bitrate is positive

    # Compress the video using ffmpeg with specified settings
    (
        ffmpeg
        .input(input_path)
        .output(output_path, 
                video_bitrate=f'{int(final_bitrate / 1024)}k',  # Convert to kbps
                preset='medium',  # Change to fast or slow as needed
                vcodec='libx264',  # Use H.264 codec for compatibility
                acodec='aac',      # Use AAC audio codec
                audio_bitrate='128k',  # Set audio bitrate
                threads='auto',    # Utilize all available threads
                **{'movflags': 'faststart'})  # Optimize for web
        .run(overwrite_output=True)
    )

    compressed_size = os.path.getsize(output_path)
    print(f"Compressed {input_path} to {output_path}.")
    print(f"Original size: {original_size / (1024 * 1024):.2f} MB, Compressed size: {compressed_size / (1024 * 1024):.2f} MB")
    
    return "compressed", input_path, original_size, compressed_size

def compress_videos_in_directory(input_dir, output_dir, target_size=30 * 1024 * 1024):
    # Create the output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)

    compressed_count = 0
    skipped_count = 0
    total_count = 0

    # Loop over each file in the input directory
    for filename in os.listdir(input_dir):
        if filename.endswith(".mp4"):  # Add more extensions if needed
            total_count += 1
            input_path = os.path.join(input_dir, filename)
            output_path = os.path.join(output_dir, filename)
            
            # Compress the video or save it as skipped
            result = compress_video(input_path, output_path, target_size)
            if result[0] == "compressed":
                compressed_count += 1
            elif result[0] == "skipped":
                skipped_count += 1

    print(f"\nSummary:")
    print(f"Total videos: {total_count}")
    print(f"Compressed videos: {compressed_count}")
    print(f"Skipped videos: {skipped_count}")

# Paths for the input and output directories
input_directory = '/kaggle/input/unique-video-crafts'  # Input directory with video files
output_directory = '/kaggle/working/compressed_videos-crafts/'  # Output directory for compressed videos

# Compress all videos in the specified input directory
compress_videos_in_directory(input_directory, output_directory)


In [ ]:
import os
import zipfile

# Define paths
dataset_name = "gaming-compressed"  # Slug-friendly name for your dataset
folder_to_zip = "/kaggle/working/compressed_videos-crafts/"  # Path to your folder
zip_file_path = f"{folder_to_zip}/{dataset_name}.zip"  # Path for the zip file

# Create the folder to zip if it doesn't exist
os.makedirs(folder_to_zip, exist_ok=True)

# Create a zip file of the folder
with zipfile.ZipFile(zip_file_path, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, dirs, files in os.walk(folder_to_zip):
        for file in files:
            # Exclude the zip file itself if rerun
            if file != f"{dataset_name}.zip":
                zipf.write(os.path.join(root, file),
                           os.path.relpath(os.path.join(root, file), folder_to_zip))

# Confirm that the zip file was created
print(f"Zip file created: {zip_file_path}")

In [ ]:
!pip install kaggle

In [4]:
# Move the API key to the correct location
!mkdir -p ~/.kaggle
!cp /kaggle/input/jsonkag/kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json  # Ensure the API key file has the right permissions

In [ ]:
import os

# Define paths and metadata
dataset_name = "Crafts-compressed"  # Give your dataset a unique name
folder_to_upload = "/kaggle/working/compressed_videos-crafts/"  # Path to your dataset
metadata_file_path = f"{folder_to_upload}/dataset-metadata.json"  # Full path to metadata file

# Create the folder to upload if it doesn't exist
!mkdir -p $folder_to_upload

# Check if the metadata file exists
if not os.path.exists(metadata_file_path):
    # Create a metadata file for the dataset
    with open(metadata_file_path, 'w') as f:
        f.write('{\n'
                '  "title": "Crafts-compressed",\n'
                '  "id": "tamimshadman/compressed_videos-crafts",\n'
                '  "licenses": [{"name": "CC0-1.0"}]\n'
                '}')

# Check if the metadata file was created successfully
!ls {folder_to_upload}

# Create a new dataset using the Kaggle API with --dir-mode option (using zip)
!kaggle datasets create -p $folder_to_upload --dir-mode zip 
